Name : Abhijeet Pal
<br>
Roll No : 200100107
<br>
Assignment 2

#1. Download labeled faces in the wild (LFW) dataset: http://vis-www.cs.umass.edu/lfw/


In [ ]:
!tar -xvf /content/drive/MyDrive/EE782/lfw.tar
# unzipping the tar file lfw.tar

In [ ]:
# importing required libraries
import os
import random
from sklearn.model_selection import train_test_split
import cv2
import torch
import torch.nn as nn
import torchvision.models as models
from google.colab.patches import cv2_imshow
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR, CosineAnnealingLR, ExponentialLR
import numpy as np

#2. Get the number of persons who have more than one image

In [ ]:
# lists which store folders with more than one file
# and those with single files
folder_multi_files = []
folder_single_files = []
lfw_path = "/content/lfw"
def count_files(path):
  ls = []
  for file in os.listdir(path):
    # checking only image files, i.e. with extensions given below
    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
      ls.append(file)
  return len(ls)
count2 = 0
count1 = 0
for root, dirs, files in os.walk(lfw_path):
    for dir_name in dirs:
        dir_path = os.path.join(root, dir_name)
        num_count = count_files(dir_path)
        if num_count ==1:
          count1+=1
          folder_single_files.append(dir_path)
        elif num_count > 1:
          count2 += 1
          folder_multi_files.append(dir_path)
print(f"Number of folders with more than one image file: {count2}")
print(f"Number of folders with only  one image file: {count1}")
# printing the count of these files

In [ ]:
print(folder_single_files[0])

# Part A
##3.  Split into training, validation, and testing by person (not by image)

In [ ]:
# splitting into training, validation and test
# 70% for training , 15% for validation,
# 15% for testing
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15
folders = folder_multi_files
random.shuffle(folders)
# randomly shuffling before assigning to train_test_split
train_folders, rest = train_test_split(folders, test_size=val_ratio + test_ratio, random_state=12)
val_folders, test_folders = train_test_split(rest, test_size=test_ratio / (val_ratio + test_ratio), random_state=12)

In [ ]:
print(len(train_folders))
print(len(test_folders))
print(len(val_folders))
# length of the training, testing, validation sets

In [ ]:
# These are too many folders, can't go with all else will take huge computation time
# Choosing those which have between 10 to 50 images
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15
folders = []
random.shuffle(folders)
for folder in folder_multi_files:
  num = len(os.listdir(folder))
  if 9<=num<=50:
    folders.append(folder)

train_folders, rest = train_test_split(folders, test_size=val_ratio + test_ratio, random_state=12)
val_folders, test_folders = train_test_split(rest, test_size=test_ratio / (val_ratio + test_ratio), random_state=12)

In [ ]:
print(len(train_folders))
print(len(test_folders))
print(len(val_folders))
# Now, this looks good enough

# 4. Start with a network that is pre-trained on ImageNet and appropriate for your computational resources

In [ ]:
model = models.resnet18(pretrained=True)
# using a pre-trained resnet18 model

In [ ]:
def new_model():
  model = models.resnet18(pretrained=True)
  new_model = torch.nn.Sequential(*list(model.children())[:-1])
  classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(512 * 1 * 1, 1000),
  )
  final_model = nn.Sequential(
      new_model,
      classifier
  )
  return final_model

# new model where the outermost layer is removed, a small neural network fully
# connected is applied and output is limited to 1000X1

In [ ]:
model = models.resnet18(pretrained=True)
# testing in separate cells

In [ ]:
new_model = torch.nn.Sequential(*list(model.children())[:-1])
# removing last layer

In [ ]:
sample_input = torch.randn(1, 3, 224, 224)
output = new_model(sample_input)
print(output.shape)
# current output shape

In [ ]:
classifier = nn.Sequential(
  nn.Flatten(),
  nn.Linear(512 * 1 * 1, 1000),
)
# changing output to size of 1000 by inserting a linear layer

In [ ]:
final_model = nn.Sequential(
    new_model,
    classifier
)
# this is the final model

In [ ]:
sample_input = torch.randn(1, 3, 224, 224)
output = final_model(sample_input)
print(output.shape)
# testing the model on a sample random input

# 5. Appropriately crop and resize the images based on your computational resources

In [ ]:
# Finding faces in the images, so that training and the rest are good
# Image preprocessing so that model trains well
# source : https://www.analyticsvidhya.com/blog/2022/10/face-detection-using-haar-cascade-using-python/#:~:text=Haar%20Cascade%20is%20a%20feature,can%20run%20in%20real%2Dtime.
def find_face (img_path):
  image = cv2.imread(img_path)
  face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
  # converting to grayscale images face detection works better
  gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
  # Detecting faces
  faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30))
  # Draw rectangles around the detected faces
  if len(faces) > 0:
      lis = min(faces, key=lambda x: x[0])
      x, y, w, h = lis
      face = image[y:y+h, x:x+w]
      resized_face = cv2.resize(face, (224, 224))
      return resized_face
  else:
    return cv2.resize(image, (224,244))


In [ ]:
person_folder_path = train_folders[0]
img = os.listdir(person_folder_path)[0]
cv2_imshow(find_face(os.path.join(person_folder_path,img)))
# testing the function on a sample image

In [ ]:
print(train_folders[0])

In [ ]:
# Now for all the folders in the test, train and validate splits, we'll save a new dataset with the cropped images
# resized to (224,224) size
for folder in train_folders:
  person = os.path.basename(folder)
  for image in os.listdir(folder):
    img_path = os.path.join(folder,image)
    face = find_face(img_path)
    new_path = '/content/train_dataset/' +os.path.join(person,image)
    os.makedirs(os.path.dirname(new_path), exist_ok=True)
    cv2.imwrite(new_path, face)

In [ ]:
# same for testing
for folder in test_folders:
  person = os.path.basename(folder)
  for image in os.listdir(folder):
    img_path = os.path.join(folder,image)
    face = find_face(img_path)
    new_path = '/content/test_dataset/' +os.path.join(person,image)
    os.makedirs(os.path.dirname(new_path), exist_ok=True)
    cv2.imwrite(new_path, face)

In [ ]:
# same for validation
for folder in val_folders:
  person = os.path.basename(folder)
  for image in os.listdir(folder):
    img_path = os.path.join(folder,image)
    face = find_face(img_path)
    new_path = '/content/val_dataset/' +os.path.join(person,image)
    os.makedirs(os.path.dirname(new_path), exist_ok=True)
    cv2.imwrite(new_path, face)


In [ ]:
# Making new lists, for storing folder paths of train, test and validate
train_fol = []
test_fol = []
val_fol = []
for folder in os.listdir('/content/train_dataset'):
  train_fol.append('/content/train_dataset/'+ str(folder))
for folder in os.listdir('/content/test_dataset'):
  test_fol.append('/content/test_dataset/'+ str(folder))
for folder in os.listdir('/content/val_dataset'):
  val_fol.append('/content/val_dataset/' + str(folder))


In [ ]:
print(train_fol[0])

In [ ]:
img_path = '/content/train_dataset/Abdullah_Gul/Abdullah_Gul_0001.jpg'
cv2_imshow(img)
# displaying an image from the training_dataset for checking saved properly

#6 Siamese Model


## a. Use image augmentation

In [ ]:
# Image augmentation
# Horizontal Flip flips horizontally
# Rotation by 10 degrees
# Adding jitter, resize and distorting the image
augmentation_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomPerspective(distortion_scale=0.5),
    transforms.Resize(size=(224, 224), antialias=True),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
])
transform = transforms.Compose([
    transforms.ToTensor(), # converting to tensor
    augmentation_transforms,
])



## Setup with dropout (Regularization)

In [ ]:
# defining the siamese class
# model is resnet 18 last layer removed, nn added with 1x1000 output

class Siamese(nn.Module):
    def __init__(self, model, p=0.1):
        super(Siamese, self).__init__()
        self.model = model
        self.dropout = nn.Dropout(p)
        # dropout with probability p
    def forward(self, image1, image2):
        temp1 = self.model(image1)
        output1 = self.dropout(temp1)
        temp2 = self.model(image2)
        output2 = self.dropout(temp2)
        return output1, output2
        # outputs are what we get after dropout
    def similarity(self, image1, image2, threshold):
        output1, output2 = self.forward(image1, image2)
        distance = F.pairwise_distance(output1, output2)
        if distance.item() < threshold:
            return 1
        else:
            return 0
    # if the distance is less than threshold than images are same
    # else they are different

In [ ]:
# using hingle loss
class hinge_loss(nn.Module):
    def __init__(self):
        super(hinge_loss, self).__init__()

    def forward(self, output1, output2, target, margin=2.0):
        # margin is taken to be 2, finding the distance and himge loss is maximum of margin- distance, 0
        # mean with target as given below
        euclidean_distance = torch.sqrt(torch.sum(torch.pow(output1 - output2, 2), dim=1))
        loss_hinge = torch.max(torch.tensor(0.0), margin - euclidean_distance)
        loss_hinge = torch.mean(target * loss_hinge + (1 - target) * torch.pow(torch.clamp(margin - euclidean_distance, min=0.0), 2))
        return loss_hinge


In [ ]:
# defining contrastive loss
# ideal for siamese model
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=2.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        dist = F.pairwise_distance(output1, output2)
        # distance between the two outputs
        # loss is mean of target * distance square + margin-dist
        loss=torch.mean(label * torch.pow(dist, 2) + (1-label) * torch.clamp(self.margin - dist, min=0.0))
        return loss


## Dataloader for Siamese Network

In [ ]:
# defining the dataloader for the siamese network
class SiameseDataloader(Dataset):
    def __init__(self, folder_path_list, transform=None):
        self.transform = transform
        self.folders_path = folder_path_list
        # will give transforms for the image and the folder_path from which images are laoded

    def __getitem__(self, index, type = random.randint(0, 1)):
        # function which will return same images to image1
        def same_image(folders_path, index, images_list, img1 ):
          images_list = os.listdir(self.folders_path[index])
          img2_path = folders_path[index] + '/' + random.choice(images_list)
          img2 = cv2.imread(img2_path)
          img2 = self.transform(img2)
          return img1, img2, 1
        # function which returns image different than image1
        def diff_image (folders_path, index, imges_list, img1):
          sec_index = random.randint(0,len(folders_path)-1)
          if sec_index!=index:
            images_list = os.listdir(folders_path[sec_index])
            img2_path = folders_path[sec_index] + '/' + random.choice(images_list)
            img2 = cv2.imread(img2_path)
            img2 = cv2.resize(img2, (244,244))
            img2 = self.transform(img2)
          else:
            if index!=0:
              sec_index = index-1
            else:
              sec_index = index+1
            images_list = os.listdir(folders_path[sec_index])
            img2_path = folders_path[sec_index] + '/' + random.choice(images_list)
            img2 = cv2.imread(img2_path)
            img2 = cv2.resize(img2, (244,244))

            img2 = self.transform(img2)

          return img1, img2, 0

        # choosing image using the imdex given
        images_list = os.listdir(self.folders_path[index])
        img1_path = self.folders_path[index]+'/'+random.choice(images_list)
        img1 = cv2.imread(img1_path)
        img1 = cv2.resize(img1, (244,244))
        img1 = self.transform(img1)
        # if type is 1 then return same image else return different image
        if (type ==1):
          return same_image(self.folders_path, index, images_list, img1)
        else:
          return diff_image(self.folders_path, index, images_list, img1)


    def __len__(self):
        return len(self.folders_path)

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
device
# check if gpu is available, sadly not

# Training with dropout


In [ ]:
# training using dropout regularization and contrastiveloss
from torch.utils.data.distributed import Sampler
train_dataset = SiameseDataloader(train_fol, transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataset = SiameseDataloader(val_fol , transform=transform)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=True)
# making both training dataloader and validation dataloader
Siamese_network = Siamese(final_model,p=0.1)
# After experimenting with different values of p, 0.1 seems to the best
# as the training loss reduces tremendously in 1 epoch which did not happen
# for other values of p
# initializing the siamese model
criterion = ContrastiveLoss()
# using siamese loss
# criterion = hinge_loss()
optimizer = torch.optim.Adam(Siamese_network.parameters(), lr=0.001)
# adam optimizer is used
num_epochs = 10
# running for 10 epochs
train_loss = []
val_loss = []
# these lists will store training and validation loss for each epoch
for epoch in range(num_epochs):
    total_train = 0
    total_val = 0

    # Training
    Siamese_network.train()
    for _, data in enumerate(train_dataloader, 0):
        img1, img2, label = data
        optimizer.zero_grad()
        # output of the siamese network
        output1, output2 = Siamese_network(img1.to(device), img2.to(device))
        # putting that to the contrastive loss
        loss = criterion(output1.to(device), output2.to(device), label.to(device))
        # backpropagating the loss and stepping the optimizer, adding loss to the train loss
        loss.backward()
        optimizer.step()
        total_train += loss.item()

    # Validation
    Siamese_network.eval()
    with torch.no_grad():
        for _, data in enumerate(val_dataloader, 0):
            img1, img2, label = data
            # outputs for the validation data, loss calculation using contrastive loss and adding to the total validation loss
            output1, output2 = Siamese_network(img1.to(device), img2.to(device))
            loss = criterion(output1.to(device), output2.to(device), label.to(device))
            total_val += loss.item()
    # finding average loss for each epoch and printing the training and validation loss
    average_train_loss = total_train / len(train_dataloader)
    average_valid_loss = total_val / len(val_dataloader)
    print(f'Epoch [{epoch + 1}/{num_epochs}], Training Loss: {average_train_loss:.6f}, Validation Loss: {average_valid_loss:.6f}')

    train_loss.append(average_train_loss)
    val_loss.append(average_valid_loss)
    # storing the loss for each epoch in the lists for plotting later

In [ ]:
# Plot
# from 1 to number of epochs
x = range(1,num_epochs + 1)
plt.plot(x, train_loss, label='Training Loss')
plt.plot(x, val_loss, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss vs Epochs')
plt.grid()
plt.legend()
plt.show()

Observation : We see that the training and validation loss decrease with the number of epochs, the training and validation loss is too high at epoch 1, however as the number of epochs increase, the loss decreases as the siamese model is getting well trained due to the backpropagation. This is expected as the loss is expected to reduce as the number of epochs increase

#7. Experiment with at least two learning rate schedulers and comment on what works, doesn’t work, and potential reason why

In [ ]:
# two learning rate schedulers used are  StepLR and CosineAnnealing
import warnings
warnings.filterwarnings("ignore")
import numpy as np
train_dataset = SiameseDataloader(train_fol, transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# making the training dataloader
# initializing a new siamese model with p=0, here
model = new_model()
siamese_network = Siamese(model, p=0)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(siamese_network.parameters(), lr=0.01)
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)
# using StepLR scheduler with step size 10 , and gamma 0.5
num_epochs = 7
# 7 epochs are used
train_loss_slr = []  # To store training loss values for plotting
# running only for training as to save time

for epoch in range(num_epochs):
    total_train_loss = 0.0

    siamese_network.train()
    # training the model
    for i, data in enumerate(train_dataloader, 0):
        img1, img2, label = data
        optimizer.zero_grad()
        output1, output2 = siamese_network(img1.to(device), img2.to(device))
        loss = criterion(output1.to(device), output2.to(device), label.to(device))
        loss.backward()
        optimizer.step()
        # scheduler.step is used to restart the steplr learning rate scheduler
        scheduler.step()
        total_train_loss += loss.item()

    # Calculate and print the average training loss for this epoch
    average_train_loss = total_train_loss / len(train_dataloader)
    print(f'Epoch [{epoch + 1}/{num_epochs}], Training Loss: {average_train_loss:.6f}')

    # Append the average training  loss to the
    train_loss_slr.append(average_train_loss)


Observation : StepLR learning rate scheduler works on reducing the learning rate as the training progresses, as the number of epochs increase, it reduces the value of $\alpha$ so that we converge faster to the minima. However, this creases properly as we may never reach the minima if the value of $\alpha$ is reduced too much. This is visible in the plot below (comparison plot) after the next cell

In [ ]:
# defining a new model
model = new_model()
siamese_network = Siamese(model)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(siamese_network.parameters(), lr=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0.001)
# Using CosineAnnealing Learning rate scheduler

train_loss_cosine = []
# To store training loss values for plotting

for epoch in range(num_epochs):
    total_train_loss = 0.0

    # Training
    siamese_network.train()
    for i, data in enumerate(train_dataloader, 0):
        img1, img2, label = data
        optimizer.zero_grad()
        output1, output2 = siamese_network(img1.to(device), img2.to(device))
        loss = criterion(output1.to(device), output2.to(device), label.to(device))
        loss.backward()
        optimizer.step()
        # stepping the scheduler
        scheduler.step()
        total_train_loss += loss.item()

    # Calculate and print the average training loss for this epoch
    average_train_loss = total_train_loss / len(train_dataloader)
    print(f'Epoch [{epoch + 1}/{num_epochs}], Training Loss: {average_train_loss:.10f}')

    # Append the average training loss
    train_loss_cosine.append(average_train_loss)



In [ ]:
# Plotting
plt.plot(range(1, num_epochs + 1), train_loss_slr, label='StepLR')
plt.plot(range(1, num_epochs + 1), train_loss_cosine, label='CosineAnnealingLR')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Over Epochs')
plt.grid()
plt.legend()
plt.show()


#8 Experiment with at least two different optimizers and comment on what works, doesn’t work,and potential reason why


In [ ]:
# sgd and RMSprop are the two optimizers used
import torch.optim as optim
# defining a new model
model = new_model()
Siamese_network = Siamese(model)
criterion = ContrastiveLoss()

optimizer_sgd = optim.SGD(Siamese_network.parameters(), lr=0.001)
# using sgd optimizer
optimizer_rmsprop = optim.RMSprop(Siamese_network.parameters(), lr=0.001)
# using rmsprop optimizer
train_loss_sgd = []
val_loss_sgd = []
# to store training and validation loss for sgd
train_loss_rmsprop = []
val_loss_rmsprop = []
# to store training and validation loss for rmsprop
num_epochs = 7

for epoch in range(num_epochs):
    # Training with SGD optimizer
    total_train_sgd = 0
    total_val_sgd = 0
    Siamese_network.train()
    for _, data in enumerate(train_dataloader, 0):
        img1, img2, label = data
        optimizer_sgd.zero_grad()
        output1, output2 = Siamese_network(img1.to(device), img2.to(device))
        loss_sgd = criterion(output1.to(device), output2.to(device), label.to(device))
        loss_sgd.backward()
        optimizer_sgd.step()
        total_train_sgd += loss_sgd.item()

    # Validation with SGD optimizer
    Siamese_network.eval()
    with torch.no_grad():
        for _, data in enumerate(val_dataloader, 0):
            img1, img2, label = data
            output1, output2 = Siamese_network(img1.to(device), img2.to(device))
            loss_sgd = criterion(output1.to(device), output2.to(device), label.to(device))
            total_val_sgd += loss_sgd.item()

    average_train_loss_sgd = total_train_sgd / len(train_dataloader)
    average_valid_loss_sgd = total_val_sgd / len(val_dataloader)
    train_loss_sgd.append(average_train_loss_sgd)
    val_loss_sgd.append(average_valid_loss_sgd)
    # storing loss in the appropriate lists


    total_train_rmsprop = 0
    total_val_rmsprop = 0
    Siamese_network.train()
    for _, data in enumerate(train_dataloader, 0):
        img1, img2, label = data
        optimizer_rmsprop.zero_grad()
        output1, output2 = Siamese_network(img1.to(device), img2.to(device))
        loss_rmsprop = criterion(output1.to(device), output2.to(device), label.to(device))
        loss_rmsprop.backward()
        optimizer_rmsprop.step()
        total_train_rmsprop += loss_rmsprop.item()

    Siamese_network.eval()
    with torch.no_grad():
        for _, data in enumerate(val_dataloader, 0):
            img1, img2, label = data
            output1, output2 = Siamese_network(img1.to(device), img2.to(device))
            loss_rmsprop = criterion(output1.to(device), output2.to(device), label.to(device))
            total_val_rmsprop += loss_rmsprop.item()

    average_train_loss_rmsprop = total_train_rmsprop / len(train_dataloader)
    average_valid_loss_rmsprop = total_val_rmsprop / len(val_dataloader)
    train_loss_rmsprop.append(average_train_loss_rmsprop)
    val_loss_rmsprop.append(average_valid_loss_rmsprop)
    # storing loss in the appropriate lists

    print(f'Epoch [{epoch + 1}/{num_epochs}], Training Loss (SGD): {average_train_loss_sgd:.6f}, Validation Loss (SGD): {average_valid_loss_sgd:.6f}')
    print(f'Epoch [{epoch + 1}/{num_epochs}], Training Loss (RMSprop): {average_train_loss_rmsprop:.6f}, Validation Loss (RMSprop): {average_valid_loss_rmsprop:.6f}')



In [ ]:
# Plot
x = range(1,num_epochs + 1)
plt.plot(x, train_loss_sgd, label='Training Loss SGD')
plt.plot(x, train_loss_rmsprop, label='Training Loss RMSProp')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss vs Epochs')
plt.grid()
plt.legend()
plt.show()

Observation : We see that the training loss for the first epoch for rmsprop is much higher, however at the second epoch only, the loss becomes much smaller than the SGD optimizer. This can be justified by stating that the RMS prop optimizer has momentum terms, which help in reducing the loss much faster compared to normal stochastic gradient descent method

In [ ]:
# Plot
x = range(1,num_epochs + 1)
plt.plot(x, val_loss_sgd, label='Validation Loss SGD')
plt.plot(x, val_loss_rmsprop, label='Validation Loss RMSProp')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Validation Loss vs Epochs')
plt.grid()
plt.legend()
plt.show()

In [ ]:
# Plotting from epoch 1 to see differences properly
x = range(1,num_epochs + 1)
plt.plot(x[1:], val_loss_sgd[1:], label='Validation Loss SGD')
plt.plot(x[1:], val_loss_rmsprop[1:], label='Validation Loss RMSProp')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Validation Loss vs Epochs')
plt.grid()
plt.legend()
plt.show()

Observation : Even in the validation loss, we see that for epochs 2 and 3 the loss by RMS prop optimizer is much smaller than SGD. This is because of the adaptive learning rate of the rmsprop optimizer. This is mostly because of the momentum term in the rmsprop optimizer. Both the optimizers work well, however rmsprop is helpful in reducing the loss in small number of epochs which is required or appreciated

# 9. Test on the test split

In [ ]:
test_dataset = SiameseDataloader(test_fol, transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)
# creating test dataloader
total_test_loss = 0.0
# evaluating on the test dataloader
Siamese_network.eval()
with torch.no_grad():
    for i, data in enumerate(test_dataloader, 0):
        img1, img2, label = data
        output1, output2 = Siamese_network(img1.to(device), img2.to(device))
        loss = criterion(output1.to(device), output2.to(device), label.to(device))
        total_test_loss += loss.item()
# printing the total loss obtained in the test split
print(f'Test Loss : {total_test_loss}')

# 10. Gather a few images of yourself and your friends (with their consent) and check if the model works well on this data

In [ ]:
real_fol = []
# testing on the real_dataset of friends obtained
for folder in os.listdir('/content/real_dataset'):
  real_fol.append('/content/real_dataset/' + str(folder))
test_real = SiameseDataloader(real_fol, transform=transform)
real_dataloader = DataLoader(test_real, batch_size=32, shuffle=True)

total_test_loss = 0.0

Siamese_network.eval()
with torch.no_grad():
    for i, data in enumerate(real_dataloader, 0):
        img1, img2, label = data
        output1, output2 = Siamese_network(img1.to(device), img2.to(device))
        loss = criterion(output1.to(device), output2.to(device), label.to(device))
        total_test_loss += loss.item()
# printing the loss obtained here
print(f'Test Loss : {total_test_loss}')

In [ ]:
# function which prints if two images are similar or not, based on the previously trained siamese network
# Siamese_network
# function just compares with threshold and returns/prints if images are same or not
# it uses the similar function defined in Siamese class

def check_similarity(img1, img2):
  threshold = 8.5
  image1_np = np.array(img1)
  image2_np = np.array(img2)

  # Convert the loaded images to PyTorch tensors
  # source : Chatgpt
  image1_tensor = torch.tensor(image1_np)
  image2_tensor = torch.tensor(image2_np)

  # Transpose and reshape the tensors to match the Siamese network input shape
  # Chatgpt for this permutation and unsqueezing
  image1_tensor = image1_tensor.permute(2, 0, 1)  # Transpose the dimensions
  image2_tensor = image2_tensor.permute(2, 0, 1)

  # Expand the dimensions to match the batch size of 1
  image1_tensor = image1_tensor.unsqueeze(0)
  image2_tensor = image2_tensor.unsqueeze(0)

  # Convert the data type to float
  # chatgpt for converting tensor to float
  image1_tensor = image1_tensor.float()
  image2_tensor = image2_tensor.float()

  similarity_result = Siamese_network.similarity(image1_tensor, image2_tensor, threshold)

  if similarity_result == 1:
      print("Images are similar.")
  else:
      print("Images are dissimilar.")
# if same than print similar, else dissimilar

In [ ]:
# Printing the images :
import warnings
warnings.filterwarnings("ignore")

path1 = '/content/real_data/Abhijeet_1.jpg'
img1 = find_face(path1)
img1 = cv2.resize(img1, (224,224))
path2 = '/content/real_data/Abhijeet_2.jpg'
img2 = find_face(path2)
img2 = cv2.resize(img2, (224,224))
cv2_imshow(img1)
cv2_imshow(img2)
check_similarity(img1, img2)

In [ ]:
# Printing the images :
import warnings
warnings.filterwarnings("ignore")

path1 = '/content/real_data/Abhijeet_1.jpg'
img1 = find_face(path1)
img1 = cv2.resize(img1, (224,224))
path2 = '/content/real_data/Nirmal_1.jpeg'
img2 = find_face(path2)
img2 = cv2.resize(img2, (224,224))
cv2_imshow(img1)
cv2_imshow(img2)
check_similarity(img1, img2)

In [ ]:
# Printing the images :
import warnings
warnings.filterwarnings("ignore")

path1 = '/content/real_data/Nirmal_1.jpeg'
img1 = find_face(path1)
img1 = cv2.resize(img1, (224,224))
path2 = '/content/real_data/Nirmal_2.jpeg'
img2 = find_face(path2)
img2 = cv2.resize(img2, (224,224))
cv2_imshow(img1)
cv2_imshow(img2)
check_similarity(img1, img2)


Observation : model gives good predictions atleast on the images tested here. Results are based on 4 test images available

# Part B
##  Train a generative model for generating face images, using a GAN. The generator takes a Gaussian noise vector as input, and tries to output a face image, while the discriminator distinguishes between real and fake face images.

In [ ]:
# Importing necessary libaries.
# Code is inspired by the youtube video link given in the assignment statement
# Link : https://www.youtube.com/watch?v=_pIMdDWK5sc
import os
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

import matplotlib.pyplot as plt
! pip install pytorch_lightning
import pytorch_lightning as pl


random_seed = 42
torch.manual_seed(random_seed)

BATCH_SIZE=128
AVAIL_GPUS = min(1, torch.cuda.device_count())
NUM_WORKERS=int(os.cpu_count() / 2)

In [ ]:
# defining the discriminator class it uses droput layer as well
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        # Simple CNN
        self.conv1 = nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1)  # Output: [32, 32, 32]
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1)  # Output: [64, 16, 16]
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 1)
# forward layer, contains two fully connected layers and 2 dropout layers
# output is given by sigmoid layer
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv2_drop(x)
        x = x.view(-1, 64 * 16 * 16)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return torch.sigmoid(x)


In [ ]:
# generator class
# input is noise
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super(Generator, self).__init__()
        self.lin1 = nn.Linear(latent_dim, 7 * 7 * 256)
        self.ct1 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=0)  # Adjust kernel size and padding
        self.ct2 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=0)   # Adjust kernel size and padding
        self.ct3 = nn.ConvTranspose2d(64, 3, 4, stride=2, padding=3)     # Adjust kernel size and padding
        # Kernel size is adjusted (dififerent from video) to satisfy input and intermediate dimensions properly
# three channels for coloured images
    def forward(self, x):
        x = self.lin1(x)
        x = F.relu(x)
        x = x.view(-1, 256, 7, 7)
        x = self.ct1(x)
        x = F.relu(x)
        x = self.ct2(x)
        x = F.relu(x)
        x = self.ct3(x) #three for 3 channels
        return x


In [ ]:
z = torch.randn(64, 100)  # Example noise vector of size (64, 100)
generator = Generator(latent_dim=100)
fake_data = generator(z)
print(fake_data.shape)
# testing the generator

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
torch.cuda.is_available()
# sadly cuda is not available again, takes long time for training

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
# defining data transformations, resizing and converting to tensors
transform = transforms.Compose([
    transforms.Resize((64, 64)),  # Resize to a common size
    transforms.ToTensor(),
])

# Creating a custom dataset  using ImageFolder in the training dataset
train_dataset = ImageFolder('/content/train_dataset/', transform=transform)

# Define batch size and creating DataLoader
batch_size = 64
dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


In [ ]:
from torchvision.utils import save_image
# to save image

In [ ]:
batch_size = 64
learning_rate = 0.001
epochs = 25
latent_dim = 100  # Size of the noise vector

# Create directories to save generated images
os.makedirs("images3", exist_ok=True)

# Initialize the Generator and Discriminator
generator = Generator(latent_dim=100)
discriminator = Discriminator()

# Define loss functions
# using Binary cross entropy loss here
adversarial_loss = nn.BCELoss()

# Define optimizers
optimizer_G = optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))

# Training loop
for epoch in range(epochs):
    for i, data in enumerate(dataloader, 0):
        real_data, _ = data
        real_data = real_data.to(device)


        # Training the Discriminator
        optimizer_D.zero_grad()
        valid = torch.ones(real_data.size(0), 1, device=device)
        fake = torch.zeros(real_data.size(0), 1, device=device)


        # Generate noise
        z = torch.randn(real_data.size(0), latent_dim).to(device)
        fake_data = generator(z)


        # Discriminator loss on real data
        loss_real = adversarial_loss(discriminator(real_data), valid)

        # Discriminator loss on fake data
        # print(fake_data.shape)
        # print(discriminator(fake_data.detach()).shape)
        # debug lines for debugging the code

        loss_fake = adversarial_loss(discriminator(fake_data.detach()), fake)

        # Total discriminator loss
        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        # Training the Generator
        optimizer_G.zero_grad()
        loss_G = adversarial_loss(discriminator(fake_data), valid)
        loss_G.backward()
        optimizer_G.step()
    save_image(fake_data.data, "images3/epoch_%d.png" % epoch, nrow=8, normalize=True)

    print(f'Epoch [{epoch}/{epochs}], Loss D: {loss_D.item()}, Loss G: {loss_G.item()}')

In [ ]:
# Displaying some of the obtained images
img = cv2.imread('/content/images3/epoch_15.png')
cv2_imshow(img)

In [ ]:
img = cv2.imread('/content/images3/epoch_17.png')
cv2_imshow(img)

Observation : We did not obtain too good results, following the video given. Idea was to use something else, a better generator and a better discriminator using some other features. In search of this found another youtube video which showed in generating anime faces.
Implementation is below

# Did not obtain good results so trying method in the youtube video and corresponding blog
https://jovian.com/aakashns/06b-anime-dcgan

In [ ]:
# need larger dtaaset than just train_dataset above
# for well working of this model.
# Merging all train, test and validation folders now
# so that training is good
for folder in train_folders:
  person = os.path.basename(folder)
  for image in os.listdir(folder):
    img_path = os.path.join(folder,image)
    face = find_face(img_path)
    new_path = '/content/new_dataset/' +os.path.join(person,image)
    os.makedirs(os.path.dirname(new_path), exist_ok=True)
    cv2.imwrite(new_path, face)

for folder in test_folders:
  person = os.path.basename(folder)
  for image in os.listdir(folder):
    img_path = os.path.join(folder,image)
    face = find_face(img_path)
    new_path = '/content/new_dataset/' +os.path.join(person,image)
    os.makedirs(os.path.dirname(new_path), exist_ok=True)
    cv2.imwrite(new_path, face)

In [ ]:
for folder in val_folders:
  person = os.path.basename(folder)
  for image in os.listdir(folder):
    img_path = os.path.join(folder,image)
    face = find_face(img_path)
    new_path = '/content/new_dataset/' +os.path.join(person,image)
    os.makedirs(os.path.dirname(new_path), exist_ok=True)
    cv2.imwrite(new_path, face)

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
image_size = 64
batch_size = 128
stats = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
DATA_DIR = '/content/new_dataset/'
# directory is the new_dataset directory
batch_size = 64


In [ ]:
print(len(os.listdir('/content/new_dataset')))
# 172 folders are now used

In [ ]:
train_ds = ImageFolder(DATA_DIR, transform=T.Compose([
    T.Resize(image_size),
    T.CenterCrop(image_size),
    T.ToTensor(),
    T.Normalize(*stats)]))
# the dataset and dataloader are defined
train_dl = DataLoader(train_ds, batch_size, shuffle=True, num_workers=2, pin_memory=True)

In [ ]:
def denorm(img_tensors):
    return img_tensors * stats[1][0] + stats[0][0]

# de normalizing the image tensors

In [ ]:
import torch
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
%matplotlib inline
# libraries for plotting

In [ ]:
def show_images(images, nmax=64):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xticks([]); ax.set_yticks([])
    ax.imshow(make_grid(denorm(images.detach()[:nmax]), nrow=8).permute(1, 2, 0))

def show_batch(dl, nmax=64):
    for images, _ in dl:
        show_images(images, nmax)
        break
# displaying some images in a grid

In [ ]:
show_batch(train_dl)
# displaying the images

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
def to_device(data, device):
    if isinstance(data, (list,tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

In [ ]:
# device dataloader
class DeviceDataLoader():
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device

    def __iter__(self):
        for b in self.dl:
            yield to_device(b, self.device)

    def __len__(self):
        return len(self.dl)

In [ ]:
train_dl = DeviceDataLoader(train_dl, device)
import torch.nn as nn


In [ ]:
# discriminator defined having 4 convolution layers, each combined batch norm and leaky relu
discriminator = nn.Sequential(
    # in: 3 x 64 x 64

    nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(64),
    nn.LeakyReLU(0.2, inplace=True),
    # out: 64 x 32 x 32

    nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(128),
    nn.LeakyReLU(0.2, inplace=True),
    # out: 128 x 16 x 16

    nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(256),
    nn.LeakyReLU(0.2, inplace=True),
    # out: 256 x 8 x 8

    nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(512),
    nn.LeakyReLU(0.2, inplace=True),
    # out: 512 x 4 x 4

    nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0, bias=False),
    # out: 1 x 1 x 1

    nn.Flatten(),
    nn.Sigmoid())

In [ ]:
discriminator = to_device(discriminator, device)
latent_size = 128
# initializing the discriminator

In [ ]:
# defining the generator having 5 transpose convolution layers combined with batch norma and relu
generator = nn.Sequential(
    # in: latent_size x 1 x 1

    nn.ConvTranspose2d(latent_size, 512, kernel_size=4, stride=1, padding=0, bias=False),
    nn.BatchNorm2d(512),
    nn.ReLU(True),
    # out: 512 x 4 x 4

    nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(256),
    nn.ReLU(True),
    # out: 256 x 8 x 8

    nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(128),
    nn.ReLU(True),
    # out: 128 x 16 x 16

    nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(64),
    nn.ReLU(True),
    # out: 64 x 32 x 32

    nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1, bias=False),
    nn.Tanh()
    # out: 3 x 64 x 64
)

In [ ]:
xb = torch.randn(batch_size, latent_size, 1, 1) # random latent tensors
fake_images = generator(xb)
print(fake_images.shape)
show_images(fake_images)
# showing noise

In [ ]:
generator = to_device(generator, device)
# initializing generator in available device

In [ ]:
# function for training discriminator
def train_discriminator(real_images, opt_d):
    # Clear discriminator gradients
    opt_d.zero_grad()

    # Pass real images through discriminator
    real_preds = discriminator(real_images)
    real_targets = torch.ones(real_images.size(0), 1, device=device)
    real_loss = F.binary_cross_entropy(real_preds, real_targets)
    real_score = torch.mean(real_preds).item()

    # Generate fake images
    latent = torch.randn(batch_size, latent_size, 1, 1, device=device)
    fake_images = generator(latent)

    # Pass fake images through discriminator
    fake_targets = torch.zeros(fake_images.size(0), 1, device=device)
    fake_preds = discriminator(fake_images)
    fake_loss = F.binary_cross_entropy(fake_preds, fake_targets)
    fake_score = torch.mean(fake_preds).item()

    # Update discriminator weights
    loss = real_loss + fake_loss
    loss.backward()
    opt_d.step()
    return loss.item(), real_score, fake_score

In [ ]:
# function for training the generator
def train_generator(opt_g):
    # Clear generator gradients
    opt_g.zero_grad()

    # Generate fake images
    latent = torch.randn(batch_size, latent_size, 1, 1, device=device)
    fake_images = generator(latent)

    # Try to fool the discriminator
    preds = discriminator(fake_images)
    targets = torch.ones(batch_size, 1, device=device)
    loss = F.binary_cross_entropy(preds, targets)

    # Update generator weights
    loss.backward()
    opt_g.step()

    return loss.item()

In [ ]:
# saving images in generated folder
from torchvision.utils import save_image
sample_dir = 'generated'
os.makedirs(sample_dir, exist_ok=True)
def save_samples(index, latent_tensors, show=True):
    fake_images = generator(latent_tensors)
    fake_fname = 'generated-images-{0:0=4d}.png'.format(index)
    save_image(denorm(fake_images), os.path.join(sample_dir, fake_fname), nrow=8)
    print('Saving', fake_fname)
    if show:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.set_xticks([]); ax.set_yticks([])
        ax.imshow(make_grid(fake_images.cpu().detach(), nrow=8).permute(1, 2, 0))

fixed_latent = torch.randn(64, latent_size, 1, 1, device=device)

In [ ]:
# stores the generator and discriminator losses, displays them
def fit(epochs, lr, start_idx=1):
    torch.cuda.empty_cache()

    # Losses & scores
    losses_g = []
    losses_d = []
    real_scores = []
    fake_scores = []

    # Create optimizers
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))

    for epoch in range(epochs):
        for real_images, _ in tqdm(train_dl):
            # Train discriminator
            loss_d, real_score, fake_score = train_discriminator(real_images, opt_d)
            # Train generator
            loss_g = train_generator(opt_g)

        # Record losses & scores
        losses_g.append(loss_g)
        losses_d.append(loss_d)
        real_scores.append(real_score)
        fake_scores.append(fake_score)

        # Log losses & scores (last batch)
        print("Epoch [{}/{}], loss_g: {:.4f}, loss_d: {:.4f}, real_score: {:.4f}, fake_score: {:.4f}".format(
            epoch+1, epochs, loss_g, loss_d, real_score, fake_score))

        # Save generated images
        save_samples(epoch+start_idx, fixed_latent, show=False)

    return losses_g, losses_d, real_scores, fake_scores

In [ ]:
from tqdm.notebook import tqdm
import torch.nn.functional as F
lr = 0.0002
epochs = 10
history = fit(epochs, lr)
# fitting data onto the model

In [ ]:
# Displaying some of the images obtained
path = '/content/generated'
images = os.listdir(path)
count = 0
for image in images:
  count+=1
  if count>=8:
    img_path = path+'/'+image
    img = cv2.imread(img_path)
    cv2_imshow(img)

Observation : The images generated are better but still not great, using keras to see if something better can ge benerated

## Trying Keras Tensorflow also as images are not too good

Inspiration : https://towardsdatascience.com/gan-by-example-using-keras-on-tensorflow-backend-1a6d515a60d0

In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import os
from matplotlib import pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LeakyReLU, Conv2DTranspose, Conv2D, Reshape, Flatten, Dropout
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.models import Model

In [ ]:
# appending all training images int the list below
images = []
IMAGES_COUNT = 10000
ORIG_WIDTH = 178
ORIG_HEIGHT = 208
diff = (ORIG_HEIGHT - ORIG_WIDTH) // 2
WIDTH = 128
HEIGHT = 128
crop_rect = (0, diff, ORIG_WIDTH, ORIG_HEIGHT - diff)
for folder in train_fol[:30]:
  PIC_DIR = folder+'/'
  for pic_file in tqdm(os.listdir(PIC_DIR)[:IMAGES_COUNT]):
      pic = Image.open(PIC_DIR + pic_file).crop(crop_rect)
      pic.thumbnail((WIDTH, HEIGHT), Image.ANTIALIAS)
      images.append(np.uint8(pic)) #Normalize the images

images = np.array(images) / 255
images.shape

In [ ]:
LATENT_DIM = 32
CHANNELS = 3
# creating generator with dense, leaky relu and reshaping appropriately
def create_generator():
    gen_input = Input(shape=(LATENT_DIM, ))

    x = Dense(128 * 16 * 16)(gen_input)
    x = LeakyReLU()(x)
    x = Reshape((16, 16, 128))(x)

    x = Conv2D(256, 5, padding='same')(x)
    x = LeakyReLU()(x)

    x = Conv2DTranspose(256, 4, strides=2, padding='same')(x)
    x = LeakyReLU()(x)

    x = Conv2DTranspose(256, 4, strides=2, padding='same')(x)
    x = LeakyReLU()(x)

    x = Conv2DTranspose(256, 4, strides=2, padding='same')(x)
    x = LeakyReLU()(x)

    x = Conv2D(512, 5, padding='same')(x)
    x = LeakyReLU()(x)
    x = Conv2D(512, 5, padding='same')(x)
    x = LeakyReLU()(x)
    x = Conv2D(CHANNELS, 7, activation='tanh', padding='same')(x)

    generator = Model(gen_input, x)
    return generator

In [ ]:
# defining discriminator using convolution layer, leaky relu
def create_discriminator():
    disc_input = Input(shape=(HEIGHT, WIDTH, CHANNELS))

    x = Conv2D(256, 3)(disc_input)
    x = LeakyReLU()(x)

    x = Conv2D(256, 4, strides=2)(x)
    x = LeakyReLU()(x)

    x = Conv2D(256, 4, strides=2)(x)
    x = LeakyReLU()(x)

    x = Conv2D(256, 4, strides=2)(x)
    x = LeakyReLU()(x)

    x = Conv2D(256, 4, strides=2)(x)
    x = LeakyReLU()(x)

    x = Flatten()(x)
    x = Dropout(0.4)(x)

    x = Dense(1, activation='sigmoid')(x)
    discriminator = Model(disc_input, x)
    optimizer = RMSprop(learning_rate=0.0001, clipvalue=1.0)

    discriminator.compile(
        optimizer=optimizer,
        loss='binary_crossentropy'
    )

    return discriminator

In [ ]:
# initializaing generator and discriminator
generator = create_generator()
discriminator = create_discriminator()
discriminator.trainable = False
gan_input = Input(shape=(LATENT_DIM, ))
gan_output = discriminator(generator(gan_input))
gan = Model(gan_input, gan_output)#Adversarial Model
optimizer = RMSprop(learning_rate=0.0001, clipvalue=1.0)
gan.compile(optimizer=optimizer, loss='binary_crossentropy')

In [ ]:
# training loop and saving files in res2 folder
# named generated_number.png
import time
LATENT_DIM = 32
CHANNELS = 3
iters = 1
# only one iteration as it takes nearly 40 mins to train for 1 epoch
# did not have too much patience
batch_size = 32
FILE_PATH = '%s/generated_%d.png'
RES_DIR = 'res2'
if not os.path.isdir(RES_DIR):
    os.mkdir(RES_DIR)
CONTROL_SIZE_SQRT = 6
control_vectors = np.random.normal(size=(CONTROL_SIZE_SQRT**2, LATENT_DIM)) / 2
start = 0
d_losses = []
a_losses = []
images_saved = 0
for step in range(iters):
    start_time = time.time()
    latent_vectors = np.random.normal(size=(batch_size, LATENT_DIM))
    generated = generator.predict(latent_vectors)

    real = images[start:start + batch_size]
    combined_images = np.concatenate([generated, real])
    # concatenting the labels
    labels = np.concatenate([np.ones((batch_size, 1)), np.zeros((batch_size, 1))])
    labels += .05 * np.random.random(labels.shape)
    # discriminator loss
    d_loss = discriminator.train_on_batch(combined_images, labels)
    d_losses.append(d_loss)

    latent_vectors = np.random.normal(size=(batch_size, LATENT_DIM))
    misleading_targets = np.zeros((batch_size, 1))

    a_loss = gan.train_on_batch(latent_vectors, misleading_targets)
    a_losses.append(a_loss)

    start += batch_size
    if start > images.shape[0] - batch_size:
        start = 0

    if step % 2 == 0:
        gan.save_weights('gan.h5')

        print('%d/%d: d_loss: %.4f,  a_loss: %.4f.  (%.1f sec)' % (step + 1, iters, d_loss, a_loss, time.time() - start_time))

        control_image = np.zeros((WIDTH * CONTROL_SIZE_SQRT, HEIGHT * CONTROL_SIZE_SQRT, CHANNELS))
        control_generated = generator.predict(control_vectors)
        for i in range(CONTROL_SIZE_SQRT ** 2):
            x_off = i % CONTROL_SIZE_SQRT
            y_off = i // CONTROL_SIZE_SQRT
            control_image[x_off * WIDTH:(x_off + 1) * WIDTH, y_off * HEIGHT:(y_off + 1) * HEIGHT, :] = control_generated[i, :, :, :]
        im = Image.fromarray(np.uint8(control_image * 255))
        im.save(FILE_PATH % (RES_DIR, images_saved))
        images_saved += 1

In [ ]:
# Displaying the image
img = cv2.imread('/content/res2/generated_0.png')
cv2_imshow(img)
# a great image is generated, atleast face is visible properly

# 12. (Bonus)
Modify the GAN to become a conditional GAN, where the condition itself is (features of)a real face image. The CGAN should generate another image of the same person, and you can use the Siamese network from Part-A as an additional discriminator for person matching. You will need to ensure that it does not simply show the same image as the one used for conditional
input.

## Without using Siamese network from Part A
Inspiration  :https://learnopencv.com/conditional-gan-cgan-in-pytorch-and-tensorflow/

In [ ]:
# importing libraries
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torch.nn as nn
import torch.optim as optim

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),  # Resize to a common size
    transforms.ToTensor(),
])

# Create a custom dataset using ImageFolder (assuming one folder per class)
train_dataset = ImageFolder('/content/train_dataset/', transform=transform)
print(train_dataset.classes)
batch_size = 64
dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
# class names are the names of the people as shown below

In [ ]:
num_classes = len(train_dataset.classes)
print(num_classes)

# Size of the noise vector
latent_dim = 100
os.makedirs("cgan_result", exist_ok=True)
# total 120 classes

In [ ]:
# generator class, having 5 linear layers combined with leaky relu and batch norm
class Generator(nn.Module):
    def __init__(self, latent_dim, num_classes):
        super(Generator, self).__init__()
        self.embed = nn.Embedding(num_classes, latent_dim)
        self.model = nn.Sequential(
            nn.Linear(latent_dim + latent_dim, 128),  # Adjust units to accommodate the concatenated input
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 3 * 64 * 64),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        gen_input = torch.cat((self.embed(labels), noise), -1)
        img = self.model(gen_input)
        img = img.view(img.size(0), 3, 64, 64)
        return img


In [ ]:
# discriminator class consisting of three linear layers with leaky relu
# finally a sigmoid is used
# all layers have been modified
class Discriminator(nn.Module):
    def __init__(self, num_classes):
        super(Discriminator, self).__init__()
        self.embed = nn.Embedding(num_classes, 64 * 64)
        self.model = nn.Sequential(
            nn.Linear(64*64*4, 64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(64, 32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        d_in = torch.cat((img.view(img.size(0), -1), self.embed(labels)), -1)
        validity = self.model(d_in)
        return validity


In [ ]:

generator = Generator(latent_dim=latent_dim, num_classes=num_classes)
discriminator = Discriminator(num_classes=num_classes)
# Define loss functions
adversarial_loss = nn.BCELoss()
auxiliary_loss = nn.CrossEntropyLoss()

# Define optimizers
optimizer_G = optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))


In [ ]:
epochs = 25
for epoch in range(epochs):
    for i, data in enumerate(dataloader, 0):
        real_data, labels = data
        real_data = real_data.to(device)
        labels = labels.to(device)

        # Training the Discriminator
        optimizer_D.zero_grad()
        valid = torch.ones(real_data.size(0), 1, device=device)
        fake = torch.zeros(real_data.size(0), 1, device=device)

        # Generate noise
        z = torch.randn(real_data.size(0), latent_dim).to(device)
        fake_data = generator(z, labels)

        # Discriminator loss on real data
        # print(real_data.shape)
        # print(labels.shape)
        dis = discriminator(real_data,labels)
        loss_real = adversarial_loss(discriminator(real_data, labels), valid)

        # Discriminator loss on fake data
        loss_fake = adversarial_loss(discriminator(fake_data.detach(), labels), fake)

        # Total discriminator loss
        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        # Training the Generator
        optimizer_G.zero_grad()
        loss_G = adversarial_loss(discriminator(fake_data, labels), valid)
        loss_G.backward()
        optimizer_G.step()

    save_image(fake_data.data, "cgan_result/epoch_%d.png" % epoch, nrow=8, normalize=True)

    print(f'Epoch [{epoch}/{epochs}], Loss D: {loss_D.item()}, Loss G: {loss_G.item()}')

In [ ]:
# displaying original image and those generated for each person
folder = train_fol[0]
images = os.listdir(folder)
for i in range (5):
  print(folder)
  img  = cv2.imread(folder+'/'+ images[i])
  cv2_imshow(img)
  pred_img = cv2.imread(f'/content/cgan_result/epoch_{i}.png')
  cv2_imshow(pred_img)


In [ ]:
# Saving images appropriately with labels as name of person whose image is generated
# Helped by ChatGPT
# edited the code given by chatgpt to save images like this
import os
import torchvision.utils as vutils
num_classes = len(train_dataset.classes)# Directory to save images
save_dir = "final_cgan_output"
latent_dim = 100
# Create the directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Set the number of images you want to generate
num_images = 64

# Loop through each label
for label in range(num_classes):  # Replace num_classes with the number of classes
    # Create a subdirectory for the label
    label_dir = os.path.join(save_dir, f"{train_dataset.classes[label]}")
    os.makedirs(label_dir, exist_ok=True)

    # Generate random noise for the current label
    noise = torch.randn(num_images, latent_dim).to(device)
    class_labels = torch.full((num_images,), label, dtype=torch.long).to(device)

    # Generate conditional images for the current label
    generated_images = generator(noise, class_labels)

    # Save the generated images for the current label
    vutils.save_image(generated_images, os.path.join(label_dir, f"{train_dataset.classes[label]}_images.png"), nrow=8, normalize=True)


In [ ]:
# showing images of original person and those generated for him
img_generated = cv2.imread('/content/final_cgan_output/Alejandro_Toledo/Alejandro_Toledo_images.png')
cv2_imshow(img_generated)
original_path = '/content/lfw/Alejandro_Toledo/Alejandro_Toledo_0001.jpg'
original_img = cv2.imread('/content/lfw/Alejandro_Toledo/Alejandro_Toledo_0001.jpg')
cv2_imshow(find_face(original_path))

In [ ]:
img_generated = cv2.imread('/content/final_cgan_output/Bill_Gates/Bill_Gates_images.png')
cv2_imshow(img_generated)
original_path = '/content/lfw/Bill_Gates/Bill_Gates_0001.jpg'
cv2_imshow(find_face(original_path))
# time for Bill Gates

#Using Siamese model of part A as the discriminator now

In [ ]:
# here everything is similar to above part, except now we are using the siamese model
# as a discriminator
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torch.nn as nn
import torch.optim as optim

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),  # Resize to a common size
    transforms.ToTensor(),
])

# Create a custom dataset using ImageFolder (assuming one folder per class)
train_dataset = ImageFolder('/content/train_dataset/', transform=transform)
print(train_dataset.classes)
batch_size = 64
dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

In [ ]:
num_classes = len(train_dataset.classes)
print(num_classes)

# Size of the noise vector
latent_dim = 100
os.makedirs("cgan_new_result", exist_ok=True)

In [ ]:
# generator similar to the above method used for CGAN
class Generator(nn.Module):
    def __init__(self, latent_dim, num_classes):
        super(Generator, self).__init__()
        self.embed = nn.Embedding(num_classes, latent_dim)
        self.model = nn.Sequential(
            nn.Linear(latent_dim + latent_dim, 128),  # Adjust units to accommodate the concatenated input
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 3 * 64 * 64),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        gen_input = torch.cat((self.embed(labels), noise), -1)
        img = self.model(gen_input)
        img = img.view(img.size(0), 3, 64, 64)
        return img


In [ ]:
# Siamese class will work as Discriminator now
class Siamese(nn.Module):
    def __init__(self, model, p=0):
        super(Siamese, self).__init__()
        self.model = model
        self.dropout = nn.Dropout(p)

    def forward(self, image1, image2):
        temp1 = self.model(image1)
        output1 = self.dropout(temp1)
        temp2 = self.model(image2)
        output2 = self.dropout(temp2)
        return output1, output2

    def similarity(self, image1, image2, threshold):
        image1.requires_grad_()
        image2.requires_grad_()
        output1, output2 = self.forward(image1, image2)
        distances = F.pairwise_distance(output1, output2)

        # Create a binary mask tensor based on the condition
        similarity_mask = (distances < threshold).float()  # 1 where condition is True, 0 otherwise
        similarity_mask = similarity_mask.mean(dim=1, keepdim=True)
        return similarity_mask

In [ ]:
learning_rate = 0.001
generator = Generator(latent_dim=latent_dim, num_classes=num_classes)
model = new_model()

discriminator = Siamese(model)

# Define loss functions
adversarial_loss = nn.BCELoss()
auxiliary_loss = nn.CrossEntropyLoss()

# Define optimizers
optimizer_G = optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
from torchvision.utils import save_image


In [ ]:
epochs = 5
for epoch in range(epochs):
    for i, data in enumerate(dataloader, 0):
        real_data, labels = data
        real_data = real_data.to(device)
        labels = labels.to(device)

        # Training the Discriminator
        optimizer_D.zero_grad()
        valid = torch.ones(64, 1, device=device)
        fake = torch.zeros(64, 1, device=device)

        # Generate noise
        z = torch.randn(real_data.size(0), latent_dim).to(device)
        fake_data = generator(z, labels)
        dis = discriminator(real_data, fake_data)

        threshold = 8.5
        threshold = torch.full((64, 1), threshold)

        # Convert the data type to float
        real_data_tensor = real_data.float()
        fake_data_tensor = fake_data.float()


        similarity_result = discriminator.similarity(real_data_tensor, fake_data_tensor, threshold)
        # For the discriminator loss
        loss_real = adversarial_loss(similarity_result, valid)
        loss_fake = adversarial_loss(similarity_result, fake)

        # For the generator loss
        loss_G = adversarial_loss(similarity_result, valid)


        # Total discriminator loss
        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        # Training the Generator
        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

    save_image(fake_data.data, "cgan_new_result/epoch_%d.png" % epoch, nrow=8, normalize=True)

    print(f'Epoch [{epoch}/{epochs}], Loss D: {loss_D.item()}, Loss G: {loss_G.item()}')

In [ ]:
# However, the results obtained are too poor
img = cv2.imread('/content/cgan_new_result/epoch_4.png')
cv2_imshow(img)

Observation : The image obtained using Siamese as discriminator are extremely bad. No face is visible. Maybe the siamese model does not work well as a discriminator. The image obtained looks like some gaussian noise plotted

# References
1. Transformations : https://pytorch.org/vision/0.15/transforms.html#:~:text=Most%20transformations%20accept%20both%20PIL,accept%20batches%20of%20tensor%20images.
2. Face detection : https://www.analyticsvidhya.com/blog/2022/10/face-detection-using-haar-cascade-using-python/#:~:text=Haar%20Cascade%20is%20a%20feature,can%20run%20in%20real%2Dtime.
3. GAN method 1 : https://www.youtube.com/watch?v=_pIMdDWK5sc
4. GAN method 2 : https://jovian.com/aakashns/06b-anime-dcgan
5. GAN method 3 : https://towardsdatascience.com/gan-by-example-using-keras-on-tensorflow-backend-1a6d515a60d0
6. CGAN Method 1 : https://learnopencv.com/conditional-gan-cgan-in-pytorch-and-tensorflow/